# External Validation

This notebook evaluates the final prediction model developed using Dataset B on an independent external dataset (EFL).

- Development dataset: Dataset B
- External validation dataset: EFL
- Predictors: CON_score and SAT_score
- Target: CI_score
- Final model: Linear Regression

The EFL dataset is used exclusively for external validation.

No model training, model selection, or hyperparameter tuning is performed using the EFL dataset.

In [1]:
# در این سلول کتابخانه‌های موردنیاز و مسیرهای Notebook 4 به‌صورت خودکار تعریف می‌شوند.

from pathlib import Path
import os
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# مسیر فعلی اجرای Notebook
CURRENT_DIR = Path.cwd().resolve()


# شناسایی خودکار ریشه پروژه
if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = CURRENT_DIR.parent

elif (
    (CURRENT_DIR / "notebooks").exists()
    and (CURRENT_DIR / "data").exists()
):

    PROJECT_ROOT = CURRENT_DIR

else:

    raise FileNotFoundError(
        "Project root could not be detected. "
        "Please run this notebook from the project folder or the notebooks folder."
    )


EFL_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "dataset_EFL_model_ready.csv"
)

FINAL_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "Final_Linear_Regression_Model.joblib"
)

METADATA_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "Final_Model_Metadata.json"
)

TABLES_FOLDER = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "tables"
)

FIGURES_FOLDER = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "figures"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nEFL model-ready file:")
print(EFL_DATA_PATH)

print("\nFinal model:")
print(FINAL_MODEL_PATH)

print("\nMetadata:")
print(METADATA_PATH)


print("\nPath check:")
print(
    "EFL data exists:",
    os.path.exists(EFL_DATA_PATH)
)

print(
    "Final model exists:",
    os.path.exists(FINAL_MODEL_PATH)
)

print(
    "Metadata exists:",
    os.path.exists(METADATA_PATH)
)

Project root:
F:\E_Learning_Continuance_ML

EFL model-ready file:
F:\E_Learning_Continuance_ML\data\processed\dataset_EFL_model_ready.csv

Final model:
F:\E_Learning_Continuance_ML\models\Final_Linear_Regression_Model.joblib

Metadata:
F:\E_Learning_Continuance_ML\models\Final_Model_Metadata.json

Path check:
EFL data exists: True
Final model exists: True
Metadata exists: True


In [2]:
# در این سلول دیتاست EFL، مدل نهایی و اطلاعات Metadata مدل بارگذاری می‌شوند.

df_EFL = pd.read_csv(
    EFL_DATA_PATH
)

final_model = joblib.load(
    FINAL_MODEL_PATH
)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    model_metadata = json.load(file)

print("EFL dataset shape:")
print(df_EFL.shape)

print("\nEFL columns:")
print(df_EFL.columns.tolist())

print("\nLoaded model:")
print(type(final_model).__name__)

print("\nModel predictors:")
print(model_metadata["predictors"])

print("\nModel target:")
print(model_metadata["target"])

EFL dataset shape:
(435, 6)

EFL columns:
['Dataset', 'CON_score', 'SAT_score', 'CI_score', 'Straight_Line_Flag', 'Exact_Duplicate_Flag']

Loaded model:
LinearRegression

Model predictors:
['CON_score', 'SAT_score']

Model target:
CI_score


In [3]:
# در این سلول سازگاری ستون‌های EFL با مدل نهایی و وجود مقادیر گمشده بررسی می‌شود.

feature_columns = model_metadata["predictors"]

target_column = model_metadata["target"]

required_columns = (
    feature_columns
    + [target_column]
)

print("Required columns:")
print(required_columns)

print("\nMissing values:")
print(
    df_EFL[
        required_columns
    ].isna().sum()
)

print("\nFeature statistics:")
print(
    df_EFL[
        feature_columns
    ].agg(
        ["min", "max", "mean"]
    )
)

print("\nTarget statistics:")
print(
    df_EFL[
        target_column
    ].agg(
        ["min", "max", "mean"]
    )
)

Required columns:
['CON_score', 'SAT_score', 'CI_score']

Missing values:
CON_score    0
SAT_score    0
CI_score     0
dtype: int64

Feature statistics:
      CON_score  SAT_score
min    1.666667   1.666667
max    5.000000   5.000000
mean   3.803831   3.855939

Target statistics:
min     1.333333
max     5.000000
mean    3.887356
Name: CI_score, dtype: float64


In [4]:
# در این سلول مدل نهایی بدون هیچ آموزش مجدد روی Dataset EFL پیش‌بینی انجام می‌دهد.

X_external = df_EFL[
    feature_columns
]

y_external = df_EFL[
    target_column
]

external_predictions = final_model.predict(
    X_external
)

print("Number of external predictions:")
print(len(external_predictions))

print("\nFirst 10 predictions:")
print(
    np.round(
        external_predictions[:10],
        4
    )
)

Number of external predictions:
435

First 10 predictions:
[4.6857 4.8134 4.8134 4.8134 4.8134 4.8134 4.6157 4.8134 4.8134 4.8134]


In [5]:
# در این سلول MAE، RMSE و R² مدل نهایی روی دیتاست مستقل EFL محاسبه می‌شوند.

external_mae = mean_absolute_error(
    y_external,
    external_predictions
)

external_rmse = np.sqrt(
    mean_squared_error(
        y_external,
        external_predictions
    )
)

external_r2 = r2_score(
    y_external,
    external_predictions
)

print("External Validation Performance:")

print(
    "MAE :",
    round(external_mae, 4)
)

print(
    "RMSE:",
    round(external_rmse, 4)
)

print(
    "R²  :",
    round(external_r2, 4)
)

External Validation Performance:
MAE : 0.2235
RMSE: 0.3573
R²  : 0.5705


In [6]:
# در این سلول نتایج اصلی External Validation روی تمام 435 نمونه EFL ذخیره می‌شوند.

external_validation_results = pd.DataFrame({

    "Evaluation": [
        "External Validation - EFL"
    ],

    "Sample_Size": [
        len(df_EFL)
    ],

    "MAE": [
        external_mae
    ],

    "RMSE": [
        external_rmse
    ],

    "R2": [
        external_r2
    ]
})

external_validation_results[
    ["MAE", "RMSE", "R2"]
] = external_validation_results[
    ["MAE", "RMSE", "R2"]
].round(4)

display(
    external_validation_results
)


external_results_path = os.path.join(
    TABLES_FOLDER,
    "External_Validation_Primary_Performance.xlsx"
)

external_validation_results.to_excel(
    external_results_path,
    index=False
)

print("\nExternal validation results saved:")
print(external_results_path)

,Evaluation,Sample_Size,MAE,RMSE,R2
0,External Validation - EFL,435,0.2235,0.3573,0.5705



External validation results saved:
F:\E_Learning_Continuance_ML\outputs\tables\External_Validation_Primary_Performance.xlsx


In [7]:
# در این سلول عملکرد مدل منتخب در Test داخلی Dataset B و External Validation دیتاست EFL مقایسه می‌شود.

internal_external_comparison = pd.DataFrame({

    "Evaluation": [
        "Dataset B - Internal Test",
        "EFL - External Validation"
    ],

    "MAE": [
        0.3254,
        external_mae
    ],

    "RMSE": [
        0.5315,
        external_rmse
    ],

    "R2": [
        0.5941,
        external_r2
    ]
})

internal_external_comparison[
    ["MAE", "RMSE", "R2"]
] = internal_external_comparison[
    ["MAE", "RMSE", "R2"]
].round(4)

display(
    internal_external_comparison
)


comparison_path = os.path.join(
    TABLES_FOLDER,
    "Internal_vs_External_Performance.xlsx"
)

internal_external_comparison.to_excel(
    comparison_path,
    index=False
)

print("\nInternal vs external comparison saved:")
print(comparison_path)

,Evaluation,MAE,RMSE,R2
0,Dataset B - Internal Test,0.3254,0.5315,0.5941
1,EFL - External Validation,0.2235,0.3573,0.5705



Internal vs external comparison saved:
F:\E_Learning_Continuance_ML\outputs\tables\Internal_vs_External_Performance.xlsx


In [8]:
# در این سلول تعداد نمونه‌های علامت‌گذاری‌شده از نظر Straight-lining و Exact Duplicate در Dataset EFL بررسی می‌شود.

print("Total EFL samples:")
print(len(df_EFL))

print("\nStraight-Line Flag:")
print(
    df_EFL["Straight_Line_Flag"]
    .value_counts(dropna=False)
)

print("\nExact-Duplicate Flag:")
print(
    df_EFL["Exact_Duplicate_Flag"]
    .value_counts(dropna=False)
)


flag_combination = pd.crosstab(
    df_EFL["Straight_Line_Flag"],
    df_EFL["Exact_Duplicate_Flag"],
    margins=True
)

print("\nCombination of quality flags:")
display(flag_combination)

Total EFL samples:
435

Straight-Line Flag:
Straight_Line_Flag
False    327
True     108
Name: count, dtype: int64

Exact-Duplicate Flag:
Exact_Duplicate_Flag
False    289
True     146
Name: count, dtype: int64

Combination of quality flags:


Exact_Duplicate_Flag,False,True,All
Straight_Line_Flag,,,
False,264,63,327
True,25,83,108
All,289,146,435


In [9]:
# در این سلول پایداری External Validation با حذف پاسخ‌های پرریسک و سپس با استفاده از پاسخ‌های بدون Flag بررسی می‌شود.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib


model_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "Final_Linear_Regression_Model.joblib"
)

final_model = joblib.load(
    model_path
)


feature_columns = [
    "CON_score",
    "SAT_score"
]

target_column = "CI_score"


# سناریوی 1: حذف فقط پاسخ‌هایی که هم‌زمان هر دو Flag را دارند
high_risk_mask = ~(
    df_EFL["Straight_Line_Flag"]
    & df_EFL["Exact_Duplicate_Flag"]
)

df_sensitivity_high_risk = df_EFL[
    high_risk_mask
].copy()


# سناریوی 2: فقط پاسخ‌هایی که هیچ Flag ندارند
clean_only_mask = (
    (~df_EFL["Straight_Line_Flag"])
    & (~df_EFL["Exact_Duplicate_Flag"])
)

df_sensitivity_clean = df_EFL[
    clean_only_mask
].copy()


print(
    "Primary sample size:",
    len(df_EFL)
)

print(
    "After removing High-Risk cases:",
    len(df_sensitivity_high_risk)
)

print(
    "Clean-only sample size:",
    len(df_sensitivity_clean)
)


sensitivity_datasets = {

    "Primary_All_EFL":
        df_EFL,

    "Sensitivity_Remove_High_Risk":
        df_sensitivity_high_risk,

    "Sensitivity_Clean_Only":
        df_sensitivity_clean
}


sensitivity_results = []


for scenario_name, scenario_data in sensitivity_datasets.items():

    X_sensitivity = scenario_data[
        feature_columns
    ]

    y_sensitivity = scenario_data[
        target_column
    ]


    predictions = final_model.predict(
        X_sensitivity
    )


    mae = mean_absolute_error(
        y_sensitivity,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_sensitivity,
            predictions
        )
    )

    r2 = r2_score(
        y_sensitivity,
        predictions
    )


    sensitivity_results.append({

        "Scenario":
            scenario_name,

        "N":
            len(scenario_data),

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2
    })


sensitivity_results = pd.DataFrame(
    sensitivity_results
)


sensitivity_results[
    ["MAE", "RMSE", "R2"]
] = sensitivity_results[
    ["MAE", "RMSE", "R2"]
].round(4)


display(
    sensitivity_results
)

Primary sample size: 435
After removing High-Risk cases: 352
Clean-only sample size: 264


,Scenario,N,MAE,RMSE,R2
0,Primary_All_EFL,435,0.2235,0.3573,0.5705
1,Sensitivity_Remove_High_Risk,352,0.2677,0.3964,0.5495
2,Sensitivity_Clean_Only,264,0.2966,0.4241,0.5622


In [10]:
# در این سلول نتایج نهایی Sensitivity Analysis اعتبارسنجی خارجی ذخیره می‌شوند.

sensitivity_output_path = os.path.join(
    TABLES_FOLDER,
    "External_Validation_Sensitivity_Analysis.xlsx"
)

sensitivity_results.to_excel(
    sensitivity_output_path,
    index=False
)

print("Sensitivity analysis saved:")
print(sensitivity_output_path)

Sensitivity analysis saved:
F:\E_Learning_Continuance_ML\outputs\tables\External_Validation_Sensitivity_Analysis.xlsx


In [11]:
# در این سلول وجود فایل‌های خروجی اصلی Notebook 4 بررسی می‌شود تا از ذخیره صحیح نتایج مطمئن شویم.

expected_files = [
    "External_Validation_Primary_Performance.xlsx",
    "Internal_vs_External_Performance.xlsx",
    "External_Validation_Sensitivity_Analysis.xlsx"
]

print("Final Notebook 4 output check:\n")

for file_name in expected_files:

    file_path = os.path.join(
        TABLES_FOLDER,
        file_name
    )

    if os.path.exists(file_path):
        print("✓ Found:", file_name)
    else:
        print("✗ Missing:", file_name)

Final Notebook 4 output check:

✓ Found: External_Validation_Primary_Performance.xlsx
✓ Found: Internal_vs_External_Performance.xlsx
✓ Found: External_Validation_Sensitivity_Analysis.xlsx
